# 04.2 Debugging Playbook

The goal of this notebook is not "never make errors", but to train you to know where to start when an error appears.

Key concepts:

- shape mismatch
- dtype mismatch
- device mismatch
- numerical instability
- debugging checklist

## Learning Goals

After this notebook, you should be able to:

1. Recognize several of the most common training errors.
2. Read shape- and dtype-related error messages.
3. Use minimal but useful prints to localize problems quickly.
4. Understand why numerical instability can produce `inf
5. Build a reusable debugging workflow.

In [ ]:
import traceback

import torch
import torch.nn as nn

torch.manual_seed(42)

## One Basic Principle

When you see an error, do not immediately rewrite large parts of the code.

First answer these questions:

- what is the input shape
- what is the output shape
- what are the target shape and dtype
- are the model and tensors on the same device
- is the loss finite

In [ ]:
def print_tensor_info(name, tensor):
    print(
        f"{name}: shape={tuple(tensor.shape)}, dtype={tensor.dtype}, "
        f"device={tensor.device}, min={tensor.min().item():.4f}, max={tensor.max().item():.4f}"
    )


sample_x = torch.randn(4, 3)
sample_y = torch.tensor([0, 1, 0, 1], dtype=torch.long)
print_tensor_info("sample_x", sample_x)
print_tensor_info("sample_y", sample_y.float())

## 2. Shape Mismatch

This is one of the most common error types.

Typical causes:

- the input dimension of a `Linear` layer is wrong
- forgot to flatten
- the loss input and target shapes do not match

In [ ]:
x = torch.randn(5, 3)
broken_linear = nn.Linear(4, 2)

try:
    broken_linear(x)
except Exception as e:
    print("Caught shape mismatch / caught shape error:")
    print(type(e).__name__)
    print(e)

In [ ]:
fixed_linear = nn.Linear(3, 2)
out = fixed_linear(x)
print_tensor_info("x", x)
print_tensor_info("out", out)
print("Conclusion: first check whether the last dimension matches in_features / first check whether the last dimension matches in_features.")

## Target Dtype Mismatch

The target for `CrossEntropyLoss` is usually expected to be class indices of dtype `long`.


In [ ]:
logits = torch.randn(4, 3)
wrong_targets = torch.tensor([0.0, 1.0, 2.0, 1.0], dtype=torch.float32)
loss_fn = nn.CrossEntropyLoss()

try:
    loss_fn(logits, wrong_targets)
except Exception as e:
    print("Caught dtype mismatch / caught dtype error:")
    print(type(e).__name__)
    print(e)

In [ ]:
correct_targets = wrong_targets.long()
loss = loss_fn(logits, correct_targets)
print_tensor_info("logits", logits)
print("correct_targets dtype =", correct_targets.dtype)
print("loss =", float(loss))

## A Shape Error with BCEWithLogitsLoss

`BCEWithLogitsLoss` expects the input and target shapes to match.


In [ ]:
binary_logits = torch.randn(4, 1)
binary_targets_wrong = torch.tensor([1.0, 0.0, 1.0, 0.0])
bce_loss = nn.BCEWithLogitsLoss()

try:
    bce_loss(binary_logits, binary_targets_wrong)
except Exception as e:
    print("Caught BCE shape mismatch / caught BCE shape error:")
    print(type(e).__name__)
    print(e)

binary_targets_correct = binary_targets_wrong.unsqueeze(1)
fixed_loss = bce_loss(binary_logits, binary_targets_correct)
print("fixed_loss =", float(fixed_loss))
print("binary_logits.shape =", binary_logits.shape)
print("binary_targets_correct.shape =", binary_targets_correct.shape)

## 5. Device Mismatch

The most typical case is:

- model is on GPU
- data is on CPU

If CUDA is not available on the current machine, we only demonstrate the checking method.


In [ ]:
def assert_same_device(model, *tensors):
    model_device = next(model.parameters()).device
    for idx, tensor in enumerate(tensors):
        assert tensor.device == model_device, (
            f"tensor {idx} is on {tensor.device}, but model is on {model_device}"
        )


cpu_model = nn.Linear(3, 2)
cpu_x = torch.randn(2, 3)
assert_same_device(cpu_model, cpu_x)
print("CPU case passed / CPU case passed")

if torch.cuda.is_available():
    gpu_model = nn.Linear(3, 2).cuda()
    try:
        assert_same_device(gpu_model, cpu_x)
    except Exception as e:
        print("Simulated device mismatch / simulated device error:")
        print(type(e).__name__)
        print(e)
else:
    print("CUDA not available / CUDA is not available in the current environment, so the real device-mismatch reproduction is skipped.")

## 6. Numerical Instability

`nan loss` does not always come from the model structure itself; it can also come from unstable numerical computation.


In [ ]:
large_scores = torch.tensor([1000.0, 1001.0])
naive_softmax = torch.exp(large_scores) / torch.exp(large_scores).sum()
stable_softmax = torch.softmax(large_scores, dim=0)

print("naive_softmax =", naive_softmax)
print("contains nan / contains nan =", torch.isnan(naive_softmax).any().item())
print("stable_softmax =", stable_softmax)

In [ ]:
constant_vector = torch.tensor([3.0, 3.0, 3.0])
bad_normalized = (constant_vector - constant_vector.mean()) / constant_vector.std()
safe_normalized = (constant_vector - constant_vector.mean()) / (constant_vector.std() + 1e-6)

print("bad_normalized =", bad_normalized)
print("has nan / has nan =", torch.isnan(bad_normalized).any().item())
print("safe_normalized =", safe_normalized)

## A Practical Debug Checklist

Before and during training, you can run these quick checks:

- print the shape of one batch
- print the target dtype
- check whether logits and targets match the expected loss function
- check whether the loss is finite
- check whether gradients are `None` or all zeros

In [ ]:
debug_model = nn.Sequential(nn.Linear(3, 8), nn.ReLU(), nn.Linear(8, 2))
debug_x = torch.randn(6, 3)
debug_y = torch.tensor([0, 1, 0, 1, 1, 0], dtype=torch.long)
debug_loss_fn = nn.CrossEntropyLoss()

logits = debug_model(debug_x)
loss = debug_loss_fn(logits, debug_y)
debug_model.zero_grad()
loss.backward()

print_tensor_info("debug_x", debug_x)
print_tensor_info("logits", logits)
print("debug_y dtype =", debug_y.dtype)
print("loss is finite / loss is finite =", torch.isfinite(loss).item())

for name, param in debug_model.named_parameters():
    grad_norm = None if param.grad is None else float(param.grad.norm())
    print(f"{name}: grad_norm={grad_norm}")

In [ ]:
# Exercise 1
# If nn.Linear(16, 4) receives x.shape == (32, 8),
# what should be the first thing you check?


Exercise 1 Reference Answer

First check whether the last input dimension matches `in_features`.

Here `8 != 16`, so the linear layer input dimension is mismatched.


In [ ]:
# Exercise 2
# When should you suspect a dtype error instead of a shape error?


Exercise 2 Reference Answer

When the shapes look correct but the loss function or operator explicitly expects a specific type, check dtype first.

For example, `CrossEntropyLoss` commonly expects the target to be `long`.


## Summary

The most important outcome of this notebook is not memorizing all error messages, but building an ordered debugging habit.

I suggest checking these five things first:

1. shape
2. dtype
3. device
4. whether the loss is finite
5. whether gradients look normal